# TASK 5 — Sales Prediction Using Python

**Objective:** Predict product sales from advertising spend on TV, Radio, and Newspaper channels.

**Dataset:** The classic `Advertising.csv` dataset with columns `TV`, `Radio`, `Newspaper`, and `Sales`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from urllib.request import urlopen
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")

## 1. Load the Advertising dataset

The notebook uses a public CSV mirror of the classic dataset so it can run without a Kaggle login.

In [ ]:
DATA_URL = "https://gist.githubusercontent.com/vivek2606/f2e92da495b3cc06982b6f656bda5005/raw/advertising.csv"

df = pd.read_csv(DATA_URL)

# Normalise column names in case the source uses different capitalisation.
df.columns = [c.strip().title() for c in df.columns]

display(df.head())

## 2. Data inspection and EDA

In [ ]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)

print("\nNull values:")
print(df.isnull().sum())

print("\nDescriptive statistics:")
display(df.describe())

## 3. Pairplot of all variables

In [ ]:
sns.pairplot(df)
plt.suptitle("Advertising Spend and Sales Relationships", y=1.02)
plt.show()

## 4. Individual scatter plots

In [ ]:
channels = ["Tv", "Radio", "Newspaper"]

for channel in channels:
    plt.figure(figsize=(7, 4))
    sns.scatterplot(data=df, x=channel, y="Sales")
    plt.title(f"Sales vs {channel} Advertising Spend")
    plt.xlabel(f"{channel} Spend")
    plt.ylabel("Sales")
    plt.show()

## 5. Correlation matrix

In [ ]:
corr = df.corr(numeric_only=True)

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="Blues")
plt.title("Correlation Matrix")
plt.show()

display(corr["Sales"].sort_values(ascending=False).to_frame("Correlation with Sales"))

## 6. Train/test split

In [ ]:
X = df[["Tv", "Radio", "Newspaper"]]
y = df["Sales"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 7. Train two regression models

Linear Regression is the required baseline. Random Forest Regressor is used as the nonlinear comparison model.

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=300, random_state=42, n_jobs=-1
    )
}

results = []
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    predictions[name] = pred

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

results_df = pd.DataFrame(results).sort_values("R2", ascending=False)
display(results_df)

## 8. Select the best-performing model

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]
best_pred = predictions[best_model_name]

print("Best model:", best_model_name)
print(f"MAE:  {results_df.iloc[0]['MAE']:.4f}")
print(f"RMSE: {results_df.iloc[0]['RMSE']:.4f}")
print(f"R²:   {results_df.iloc[0]['R2']:.4f}")

## 9. Feature importance / impact

For Linear Regression, coefficients quantify the estimated change in Sales for a one-unit increase in a channel while holding the other channels constant. For Random Forest, feature importance is used.

In [ ]:
if best_model_name == "Linear Regression":
    importance = pd.Series(best_model.coef_, index=X.columns).sort_values(ascending=False)
    importance_label = "Linear Regression Coefficient"
else:
    importance = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=False)
    importance_label = "Random Forest Feature Importance"

plt.figure(figsize=(7, 4))
importance.sort_values().plot(kind="barh")
plt.title(importance_label)
plt.xlabel("Impact / Importance")
plt.ylabel("Advertising Channel")
plt.show()

print("Feature ranking:")
display(importance.to_frame("value"))

## 10. Residual plot for the best model

In [ ]:
residuals = y_test - best_pred

plt.figure(figsize=(7, 4))
sns.scatterplot(x=best_pred, y=residuals)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted Sales")
plt.ylabel("Residual (Actual - Predicted)")
plt.title(f"Residual Plot — {best_model_name}")
plt.show()

print("Mean residual:", round(residuals.mean(), 4))

## Interpretation

The feature-impact chart provides the requested answer about which advertising channel has the highest impact on Sales **for the fitted model**. The conclusion should be based on the displayed coefficients or feature importances, not on an assumed result.

A good residual plot should show points scattered around zero without a strong curve or funnel pattern. Strong systematic structure can indicate that the model is missing nonlinear relationships or other explanatory variables.

## Conclusion

This notebook completes dataset loading, null checking, descriptive statistics, pairplot, three individual scatter plots, correlation heatmap, train/test split, Linear Regression baseline, Random Forest comparison, MAE/RMSE/R² evaluation, best-model selection, feature impact analysis, and residual analysis.